# 📸 Realistic NSFW Image Generator
### RealVisXL V5.0 · No Login Required · Google Colab T4 GPU

---

## ✅ Run cells in this exact order:

| # | Cell | Notes |
|---|------|-------|
| 1 | **GPU Check** | Must show Tesla T4 |
| 2 | **Install** | Auto-restarts kernel at the end — click OK on popup |
| 3 | **Verify** | Run AFTER restart — confirms no import errors |
| 4 | **Load Model** | Downloads ~6.6 GB first time, cached after |
| 5 | **Launch UI** | Opens the image generator |

> ⚠️ After **Cell 2** a **"Session crashed" / "Kernel restarted"** popup appears — this is **completely normal**.
> Click **OK** and continue from **Cell 3**. Do NOT re-run Cell 2.

---

**Model:** RealVisXL V5.0 by SG161222  
**Why this model:** 152,000+ downloads/month · photorealistic NSFW · used in 42 active HuggingFace Spaces · confirmed diffusers format · fully public · no token needed  
**Prompting:** Natural language sentences — describe like a photographer  
**Speed:** ~60–100 sec / image on T4

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 of 5 — GPU Check                                    ║
# ║  Must show Tesla T4 below.                                   ║
# ║  If nothing: Runtime → Change runtime type → T4 GPU → Save  ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)

if r.returncode == 0 and r.stdout.strip():
    print(f'✅ GPU : {r.stdout.strip()}')
    print(f'   Python : {sys.version.split()[0]}')
    print()
    print('GPU confirmed — proceed to Cell 2!')
else:
    print('❌ No GPU detected!')
    print('   Fix: Runtime → Change runtime type → T4 GPU → Save')
    raise SystemExit('Please enable T4 GPU first.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 of 5 — Install Packages + Restart Kernel            ║
# ║                                                              ║
# ║  Installs ONLY: diffusers, accelerate, safetensors           ║
# ║  Does NOT touch: torch, transformers, gradio                 ║
# ║  (those are pre-installed in Colab at correct versions)      ║
# ║  Does NOT install xformers — causes torchvision crash        ║
# ║                                                              ║
# ║  Kernel auto-restarts after install.                         ║
# ║  "Session crashed" popup = NORMAL → click OK                 ║
# ║  Then run Cell 3 → 4 → 5. Do NOT re-run this cell.         ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

def pip(*args):
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + list(args),
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f'⚠️  pip warning: {r.stderr[-300:]}')
    return r.returncode

print('📦 Installing packages...')
print('   (Only diffusers, accelerate, safetensors)')
print('─' * 64)

pip('diffusers')    # latest — fixes all known import errors
pip('accelerate')   # required by diffusers pipeline
pip('safetensors')  # required to load .safetensors model files

print('─' * 64)
print('✅ Packages installed!')
print()
print('🔄 Restarting kernel so Python loads the new packages...')
print('   "Session crashed" popup = NORMAL — click OK')
print('   Then run Cell 3 → Cell 4 → Cell 5.')
print('   Do NOT re-run this cell after restart.')

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 of 5 — Verify (run AFTER kernel restart)            ║
# ║  Every line must show ✅ before continuing to Cell 4.       ║
# ╚══════════════════════════════════════════════════════════════╝

import sys
print(f'Python : {sys.version.split()[0]}')
print()

import torch
cuda_ok = torch.cuda.is_available()
print(f'torch         : {torch.__version__}')
print(f'CUDA available: {cuda_ok}')
if not cuda_ok:
    raise SystemExit('❌ CUDA not available. Re-enable T4 GPU: Runtime → Change runtime type → T4 GPU')
print(f'GPU           : {torch.cuda.get_device_name(0)}')
print(f'VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print()

import transformers
print(f'transformers  : {transformers.__version__}')

import diffusers
print(f'diffusers     : {diffusers.__version__}')

import gradio
print(f'gradio        : {gradio.__version__}')
print()

print('Testing pipeline imports...')
from diffusers import StableDiffusionXLPipeline
print('  ✅ StableDiffusionXLPipeline')
from diffusers import EulerAncestralDiscreteScheduler
print('  ✅ EulerAncestralDiscreteScheduler')
print()
print('=' * 56)
print('✅  ALL CHECKS PASSED — safe to run Cell 4!')
print('=' * 56)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 of 5 — Load Model                                   ║
# ║                                                              ║
# ║  Model  : RealVisXL V5.0 by SG161222                        ║
# ║  Why    : 152k+ downloads/month, confirmed diffusers format  ║
# ║           fully public, no token needed, top photorealism   ║
# ║  Size   : ~6.6 GB download (first run only, cached after)   ║
# ║  Time   : 5–10 min first run, ~60 sec cached                ║
# ╚══════════════════════════════════════════════════════════════╝

import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

MODEL_ID = 'SG161222/RealVisXL_V5.0'

print(f'📥 Loading: {MODEL_ID}')
print('First run downloads ~6.6 GB — please be patient...')
print('─' * 64)

# Load in float16 to fit T4 VRAM (15 GB)
# No variant="fp16" — repo does not publish fp16 shards,
# using that argument throws a FileNotFoundError
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)

# EulerAncestralDiscreteScheduler:
# — Universally stable across all diffusers versions
# — No index-out-of-bounds bug that DPMSolver has in newer diffusers
# — Fast, high quality, great for photorealism
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    pipe.scheduler.config
)

pipe = pipe.to('cuda')

# attention_slicing: reduces VRAM, no installs, no conflicts
# NOT using xformers — it overwrites torch and breaks torchvision on Colab
pipe.enable_attention_slicing()

# SDXL has no safety_checker — intentionally not set

print('─' * 64)
print('✅ Model loaded and ready!')
used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'🖥️  VRAM used : {used:.2f} GB')
print(f'🖥️  VRAM free : {total - used:.2f} GB')
print(f'🖥️  VRAM total: {total:.2f} GB')
print()
print('✅ Ready — run Cell 5 to open the generator!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 of 5 — Launch Image Generator UI                    ║
# ║  Gradio interface appears below this cell.                  ║
# ║  Public share link (72 hr) also printed.                    ║
# ╚══════════════════════════════════════════════════════════════╝

import gradio as gr
import torch, random, os
from datetime import datetime

LOCAL_DIR = '/content/generated_images'
os.makedirs(LOCAL_DIR, exist_ok=True)

# ── Resolutions ───────────────────────────────────────────────────
# 896×1152 is ideal for portraits on SDXL — close to native training
# resolution and excellent for solo female content
RESOLUTIONS = {
    'Portrait   896 × 1152  📱  (best for solo)' : (896, 1152),
    'Square    1024 × 1024  ⬜  (general)'       : (1024, 1024),
    'Landscape 1152 × 896   🖼️  (scene / wide)' : (1152, 896),
    'Tall       832 × 1216  📏  (full body)'      : (832, 1216),
}

# ── Default prompts ───────────────────────────────────────────────
# RealVisXL works best with natural language + photography terms
DEFAULT_POS = (
    'RAW photo, a beautiful 25-year-old caucasian woman, '
    'natural skin texture, detailed face, blue eyes, blonde hair, '
    'soft cinematic lighting, shallow depth of field, '
    'shot on Canon EOS R5 85mm f/1.4, bokeh background, '
    'hyperrealistic, ultra detailed, 8K'
)
DEFAULT_NEG = (
    'bad hands, bad anatomy, ugly, deformed, '
    'face asymmetry, eyes asymmetry, deformed eyes, deformed mouth, '
    'open mouth, cartoon, anime, illustration, painting, '
    'cgi, 3d render, blurry, watermark, signature, '
    'text, extra fingers, missing fingers, mutilated, '
    'low quality, worst quality, jpeg artifacts, plastic skin'
)

# ── Step counter using mutable list so callback can update it ─────
_step_counter = [0]
_step_total   = [30]

def _on_step(pipeline, step_index, timestep, callback_kwargs):
    _step_counter[0] = step_index + 1
    print(f'\r   🔄 Step {_step_counter[0]:>2} / {_step_total[0]}', end='', flush=True)
    return callback_kwargs

# ── Generation function ───────────────────────────────────────────
def generate(pos, neg, res_label, steps, cfg, seed_val, save_drive):
    try:
        seed = random.randint(0, 2**32 - 1) if int(seed_val) == -1 else int(seed_val)
        w, h = RESOLUTIONS[res_label]
        gen  = torch.Generator(device='cuda').manual_seed(seed)

        _step_counter[0] = 0
        _step_total[0]   = int(steps)
        print(f'🚀 Generating {w}×{h}, {int(steps)} steps, CFG {cfg}, seed {seed}...')

        out = pipe(
            prompt=pos,
            negative_prompt=neg,
            width=w,
            height=h,
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            generator=gen,
            callback_on_step_end=_on_step,
        )

        print()  # newline after step counter
        img  = out.images[0]
        ts   = datetime.now().strftime('%Y%m%d_%H%M%S')
        name = f'realvisxl_{ts}_seed{seed}.png'
        img.save(os.path.join(LOCAL_DIR, name))
        msg  = f'💾 Saved: {LOCAL_DIR}/{name}'

        drive_dir = '/content/drive/MyDrive/AI_RealVisXL'
        if save_drive:
            if os.path.exists('/content/drive/MyDrive'):
                os.makedirs(drive_dir, exist_ok=True)
                img.save(os.path.join(drive_dir, name))
                msg += f'\n📁 Drive: {drive_dir}/{name}'
            else:
                msg += (
                    '\n⚠️  Drive not mounted. Run in a new cell:\n'
                    '   from google.colab import drive\n'
                    '   drive.mount("/content/drive")'
                )

        info = f'📐 {w}×{h}   🔢 Steps: {int(steps)}   ⚙️ CFG: {cfg}   🎲 Seed: {seed}'
        return img, info, msg

    except RuntimeError as e:
        torch.cuda.empty_cache()
        if 'out of memory' in str(e).lower():
            return None, '❌ VRAM out of memory!', 'Lower Steps or switch to Portrait resolution, then retry.'
        return None, f'❌ Runtime error: {str(e)[:300]}', ''
    except Exception as e:
        return None, f'❌ Unexpected error: {str(e)[:300]}', ''


# ── Gradio UI ─────────────────────────────────────────────────────
CSS = """
.gen-btn {
    background: linear-gradient(135deg, #1a1a2e, #c94b4b) !important;
    color: #fff !important;
    font-size: 17px !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 10px !important;
    min-height: 52px !important;
}
.gen-btn:hover { opacity: 0.88 !important; }
"""

with gr.Blocks(
    title='📸 RealVisXL V5 Generator',
    theme=gr.themes.Soft(primary_hue='red', secondary_hue='slate'),
    css=CSS
) as demo:

    gr.Markdown("""
    # 📸 Realistic NSFW Generator — RealVisXL V5.0
    **Model:** RealVisXL V5.0 by SG161222 &nbsp;|&nbsp;
    **GPU:** T4 15 GB &nbsp;|&nbsp;
    **Speed:** ~60–100 sec/image &nbsp;|&nbsp;
    **No login required ✅**

    > Write prompts like describing a **photograph**. Start with `RAW photo,` and use camera/lighting terms for best realism.
    > For explicit content, add `nude`, `naked`, or `explicit` anywhere in your positive prompt.
    """)

    with gr.Row():

        # ── Left: Controls ──────────────────────────────────────
        with gr.Column(scale=5):
            pos_box = gr.Textbox(
                label='✏️  Positive Prompt  (describe what you want)',
                value=DEFAULT_POS, lines=5,
            )
            neg_box = gr.Textbox(
                label='🚫  Negative Prompt  (what to avoid)',
                value=DEFAULT_NEG, lines=3,
            )
            res_radio = gr.Radio(
                label='📐  Resolution',
                choices=list(RESOLUTIONS.keys()),
                value='Portrait   896 × 1152  📱  (best for solo)',
            )
            with gr.Row():
                steps_sl = gr.Slider(
                    label='🔢 Steps  (30 recommended)',
                    minimum=20, maximum=50, step=1, value=30
                )
                cfg_sl = gr.Slider(
                    label='⚙️ CFG Scale  (5–7 recommended)',
                    minimum=1.0, maximum=12.0, step=0.5, value=6.0
                )
            seed_box = gr.Number(
                label='🎲 Seed  (−1 = random every time)',
                value=-1, precision=0
            )
            drive_cb = gr.Checkbox(
                label='📁 Also save to Google Drive',
                value=False,
                info='Mount Drive first: from google.colab import drive; drive.mount("/content/drive")'
            )
            gen_btn = gr.Button(
                '🚀  Generate Image',
                elem_classes='gen-btn',
                variant='primary'
            )

            gr.Markdown('### 💡 Example prompts (click to load):')
            gr.Examples(
                label='',
                examples=[
                    [
                        'RAW photo, a beautiful 23-year-old caucasian woman, '
                        'long wavy blonde hair, blue eyes, natural makeup, '
                        'wearing a white sundress, standing on a hotel balcony, '
                        'golden hour sunlight, soft warm lighting, '
                        'shot on Canon EOS R5 85mm f/1.4, bokeh, hyperrealistic, 8K',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a seductive 26-year-old caucasian woman, '
                        'short brunette hair, green eyes, toned body, '
                        'wearing black lace lingerie, sitting on a bed, '
                        'soft bedroom lighting, rumpled white sheets, intimate mood, '
                        'shot on Sony A7 III 50mm f/1.8, shallow depth of field, '
                        'hyperrealistic, skin pores visible, nsfw',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a beautiful 28-year-old caucasian woman, '
                        'long red hair, hazel eyes, fit body, '
                        'fully nude, lying on white bed sheets, '
                        'natural window light, artistic nude photography, '
                        'shot on Leica M11 90mm, film grain, '
                        'hyperrealistic, extremely detailed skin, explicit',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a gorgeous 24-year-old caucasian woman, '
                        'natural blonde hair, petite body, blue eyes, '
                        'wearing a tiny bikini, poolside, summer afternoon, '
                        'wet skin glistening in sunlight, water droplets, '
                        'shot on Canon EOS 5D Mark IV, lifestyle photography, '
                        'hyperrealistic, 8K, highly detailed',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a stunning 30-year-old caucasian woman, '
                        'dark brown hair in a bun, brown eyes, curvy body, '
                        'fully nude, standing in a luxury bathroom, '
                        'steam from shower, warm ambient lighting, '
                        'shot on Nikon Z9 85mm f/1.2, cinematic lighting, '
                        'hyperrealistic, 8K resolution, explicit, nsfw',
                        DEFAULT_NEG
                    ],
                    [
                        'RAW photo, a beautiful 22-year-old caucasian woman, '
                        'long straight black hair, dark eyes, athletic body, '
                        'topless, casual jeans, leaning against a wall, '
                        'urban setting, natural daylight, candid shot, '
                        'shot on Fujifilm GFX 100S 110mm, editorial style, '
                        'hyperrealistic, highly detailed skin, nsfw',
                        DEFAULT_NEG
                    ],
                ],
                inputs=[pos_box, neg_box],
            )

        # ── Right: Output ───────────────────────────────────────
        with gr.Column(scale=5):
            out_img  = gr.Image(
                label='🖼️  Generated Image',
                type='pil', height=600
            )
            info_box = gr.Textbox(
                label='ℹ️  Generation Info',
                interactive=False, lines=1
            )
            save_box = gr.Textbox(
                label='💾  Save Status',
                interactive=False, lines=2
            )

    gen_btn.click(
        fn=generate,
        inputs=[pos_box, neg_box, res_radio, steps_sl, cfg_sl, seed_box, drive_cb],
        outputs=[out_img, info_box, save_box],
    )

    # ── Prompting Guide ─────────────────────────────────────────
    with gr.Accordion('📖  Full Prompting Guide', open=False):
        gr.Markdown("""
        ## RealVisXL V5.0 Prompting Guide

        Write prompts like briefing a **photographer** — natural sentences.
        No special tag syntax required.

        ---

        ### ⭐ Best prompt structure:
        ```
        RAW photo, [subject description], [outfit/state], [setting], [lighting], [camera info], hyperrealistic, 8K
        ```

        ### 📸 Photography terms that maximise realism:

        **Always start with:**  `RAW photo`  `analog photo`  `professional photograph`

        **Camera bodies:**
        `shot on Canon EOS R5`  `shot on Sony A7 III`  `shot on Nikon Z9`
        `shot on Leica M11`  `shot on Fujifilm GFX 100S`

        **Lenses + DOF:**
        `85mm f/1.4`  `50mm f/1.8`  `90mm f/2`
        `shallow depth of field`  `bokeh background`

        **Lighting:**
        `soft cinematic lighting`  `golden hour`  `soft window light`
        `studio lighting`  `dramatic shadows`  `warm ambient lighting`
        `rim lighting`  `natural daylight`

        **Skin / detail quality:**
        `hyperrealistic`  `extremely detailed skin`  `skin pores visible`
        `natural skin texture`  `8K`  `ultra detailed`  `film grain`

        ---

        ### 👤 Describing subjects (be specific for best results):
        - Age: `22-year-old`, `30-year-old`
        - Hair: `long wavy blonde hair`, `short brunette hair`, `red hair`
        - Eyes: `blue eyes`, `green eyes`, `hazel eyes`, `brown eyes`
        - Body: `petite body`, `athletic body`, `curvy body`, `toned body`, `fit body`
        - Face: `detailed face`, `natural makeup`, `freckles`, `full lips`

        ### 🏠 Settings:
        `bedroom`  `hotel room`  `bathroom`  `poolside`  `beach`
        `luxury apartment`  `studio`  `outdoor`  `urban street`

        ### 🔞 NSFW / Content level:
        | Add to prompt | Content |
        |---------------|---------|
        | *(nothing)* | Tasteful SFW |
        | `suggestive` | Revealing clothing |
        | `nsfw` | Adult, partial nudity |
        | `topless`, `nude`, `naked` | Topless / nude |
        | `explicit`, `nude`, `nsfw` | Fully explicit |

        ---

        ### ⚙️ Settings reference:
        | Setting | Recommended | Notes |
        |---------|-------------|-------|
        | Steps | **30** | Sweet spot — more steps = diminishing returns on T4 |
        | CFG | **5–7** | 6 is ideal; below 4 = blurry; above 9 = over-saturated |
        | Resolution | **896×1152** | Best for solo portraits |
        | Seed | -1 | Random; paste seed back to reproduce exact image |

        ### 🔁 Reproduce an image:
        Copy the seed from ℹ️ Info box → paste into Seed field → same prompt = identical image.

        ### ❌ Negative prompt tips:
        Always keep `bad anatomy, bad hands, face asymmetry, deformed` in negatives.
        Add `open mouth` if you don't want mouths open.
        Add `cartoon, anime, 3d render` to prevent non-photorealistic outputs.
        """)

    # ── VRAM Cleanup ─────────────────────────────────────────────
    with gr.Accordion('🧹  VRAM Cleanup  (if Out of Memory errors)', open=False):
        gr.Markdown("""
        Run this in a **new Colab cell**:
        ```python
        import torch, gc
        del pipe
        gc.collect()
        torch.cuda.empty_cache()
        print('VRAM cleared — re-run Cell 4 to reload the model')
        ```
        """)

    # ── Download ZIP ─────────────────────────────────────────────
    with gr.Accordion('⬇️  Download All Images as ZIP', open=False):
        gr.Markdown("""
        Run this in a **new Colab cell**:
        ```python
        import zipfile, os
        from google.colab import files
        DIR  = '/content/generated_images'
        imgs = [f for f in os.listdir(DIR) if f.endswith('.png')]
        if imgs:
            with zipfile.ZipFile('/content/realvisxl_images.zip', 'w') as zf:
                for f in imgs: zf.write(f'{DIR}/{f}', f)
            files.download('/content/realvisxl_images.zip')
            print(f'Downloaded {len(imgs)} image(s)!')
        else:
            print('No images yet — generate some first!')
        ```
        """)

print('🚀 Launching Gradio...')
demo.launch(share=True, debug=False)